# 🏥 Healthcare Data Pipeline: Patient Vitals Consumer (Bronze Layer)

**Overview**
This notebook serves as the primary data ingestion point (Consumer) for a distributed healthcare data pipeline. It securely connects to a remote Apache Kafka cluster to pull raw patient vital signs and loads the JSON payload into Databricks using Apache Spark.

**System Architecture**
* **IoT Device Simulation (Producer):** Python generator streaming telemetry data.
* **Message Broker:** Aiven Kafka (Cloud-hosted).
* **Consumer & Compute:** Databricks Apache Spark (Batch/Micro-batch processing).

**Security & Configuration**
* **Authentication:** Secure SSL/TLS connection using dynamic `.pem` and `.cert` file reads.
* **Topic:** `patient-vitals`
* **Handling:** Bypasses public DBFS checkpoint limitations by reading the topic state dynamically into memory.

---
*Data Engineering Portfolio Project*

In [0]:
import os
from pyspark.sql.functions import *
from pyspark.sql.types import *

current_dir = os.getcwd()

# 1. Read the files into Python strings
with open(f"{current_dir}/ca.pem", "r") as f:
    ca_string = f.read()

with open(f"{current_dir}/service.cert", "r") as f:
    cert_string = f.read()

with open(f"{current_dir}/service.key", "r") as f:
    key_string = f.read()

# 2. Your Aiven Credentials
kafka_bootstrap_servers = "kafka-114b7e84-rishikeshmate09-5283.l.aivencloud.com:21915"
topic = "patient-vitals"

# 3. Connect Spark to Aiven (Using a BATCH read instead of STREAM)
raw_df = spark.read \
    .format("kafka") \
    .option("kafka.bootstrap.servers", kafka_bootstrap_servers) \
    .option("subscribe", topic) \
    .option("kafka.security.protocol", "SSL") \
    .option("kafka.ssl.truststore.type", "PEM") \
    .option("kafka.ssl.keystore.type", "PEM") \
    .option("kafka.ssl.truststore.certificates", ca_string) \
    .option("kafka.ssl.keystore.certificate.chain", cert_string) \
    .option("kafka.ssl.keystore.key", key_string) \
    .option("startingOffsets", "earliest") \
    .option("endingOffsets", "latest") \
    .load()

# 4. Display the static batch of data!
display(raw_df.selectExpr("CAST(value AS STRING)"))